In [5]:
pip install scikit-learn pandas numpy matplotlib seaborn

   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   --------------------- ------------------ 4.7/8.7 MB 27.9 MB/s eta 0:00:01
   ---------------------------------------- 8.7/8.7 MB 25.9 MB/s  0:00:00
   ---------------------------------------- 0.0/38.5 MB ? eta -:--:--
   ------ --------------------------------- 6.6/38.5 MB 32.6 MB/s eta 0:00:01
   --------- ------------------------------ 9.4/38.5 MB 35.4 MB/s eta 0:00:01
   -------------------- ------------------- 19.7/38.5 MB 31.2 MB/s eta 0:00:01
   ------------------------------- -------- 30.4/38.5 MB 36.6 MB/s eta 0:00:01
   ---------------------------------------  38.3/38.5 MB 39.1 MB/s eta 0:00:01
   ---------------------------------------- 38.5/38.5 MB 35.4 MB/s  0:00:01

   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ----


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

In [7]:
!pip install scikit-learn pandas numpy matplotlib seaborn tensorflow

   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 1.6/332.0 MB 15.9 MB/s eta 0:00:21
   - -------------------------------------- 11.0/332.0 MB 35.7 MB/s eta 0:00:09
   -- ------------------------------------- 20.7/332.0 MB 38.9 MB/s eta 0:00:09
   --- ------------------------------------ 30.1/332.0 MB 40.0 MB/s eta 0:00:08
   ---- ----------------------------------- 41.4/332.0 MB 42.7 MB/s eta 0:00:07
   ----- ---------------------------------- 49.3/332.0 MB 41.7 MB/s eta 0:00:07
   ------ --------------------------------- 57.7/332.0 MB 41.1 MB/s eta 0:00:07
   ------- -------------------------------- 65.0/332.0 MB 40.8 MB/s eta 0:00:07
   -------- ------------------------------- 68.7/332.0 MB 37.4 MB/s eta 0:00:08
   --------- ------------------------------ 75.5/332.0 MB 36.7 MB/s eta 0:00:07
   ---------- ----------------------------- 87.0/332.0 MB 38.4 MB/s eta 0:00:07
   ----------- ---------------------------- 98.3/3


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import sklearn
import pandas
import numpy
print("All libraries installed successfully!")

In [ ]:
pip install --upgrade scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# ============================================================================
# LOAD AND PREPROCESS DATA
# ============================================================================

def load_and_preprocess_data(csv_file):
    """Load BME688 sensor data and preprocess for anomaly detection."""
    
    # Load CSV file
    df = pd.read_csv(csv_file)
    print(f"Loaded data shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nFirst few rows:\n{df.head()}")
    
    # Select numeric sensor columns (exclude Timestamp and ID columns)
    sensor_columns = [col for col in df.columns if col not in 
                     ['Timestamp', 'Sensor_ID', 'Sample_Index', 'Status', 'Index']]
    
    X = df[sensor_columns].values
    
    # Handle missing values
    X = np.nan_to_num(X, nan=np.nanmean(X, axis=0))
    
    # Standardize features (critical for most algorithms)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"\nFeatures used: {sensor_columns}")
    print(f"Data shape after preprocessing: {X_scaled.shape}")
    print(f"Feature scaling - Mean: {X_scaled.mean(axis=0)}, Std: {X_scaled.std(axis=0)}")
    
    return X_scaled, sensor_columns, scaler, df

In [ ]:
# ============================================================================
# METHOD 1: ISOLATION FOREST
# ============================================================================

class IsolationForestDetector:
    """Unsupervised anomaly detection using Isolation Forest."""
    
    def __init__(self, contamination=0.1):
        """
        Initialize Isolation Forest.
        contamination: Expected proportion of anomalies (0.01 to 0.5)
        """
        from sklearn.ensemble import IsolationForest
        self.contamination = contamination
        self.model = IsolationForest(contamination=contamination, 
                                     random_state=42, n_estimators=100)
        self.name = "Isolation Forest"
    
    def fit(self, X_train):
        """Train the model."""
        print(f"\n[{self.name}] Training...")
        start_time = time.time()
        self.model.fit(X_train)
        train_time = time.time() - start_time
        print(f"Training completed in {train_time:.4f} seconds")
        return train_time
    
    def predict(self, X_test):
        """Predict anomalies (-1 = anomaly, 1 = normal)."""
        start_time = time.time()
        predictions = self.model.predict(X_test)
        pred_time = time.time() - start_time
        
        # Convert to binary (0 = normal, 1 = anomaly)
        anomalies = (predictions == -1).astype(int)
        return anomalies, pred_time
    
    def get_scores(self, X_test):
        """Get anomaly scores (lower = more anomalous)."""
        scores = self.model.score_samples(X_test)
        return scores